In [23]:
import pandas as pd
import numpy as np

In [24]:
class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

def make_df(cols, ind):
    data = {c: [str(c) + str(i) for i in ind]
            for c in cols}
    return pd.DataFrame(data, ind)

# Relational Algebra

## One to one

In [41]:
df1 = make_df('ABC', range(3))
df2 = make_df('ADE', range(4, 7))
df2['A'] = df1['A'].to_numpy()
display('df1', 'df2', 'pd.merge(df1, df2)')

df1
    A   B   C
0  A0  B0  C0
1  A1  B1  C1
2  A2  B2  C2

df2
    A   D   E
4  A0  D4  E4
5  A1  D5  E5
6  A2  D6  E6

pd.merge(df1, df2)
    A   B   C   D   E
0  A0  B0  C0  D4  E4
1  A1  B1  C1  D5  E5
2  A2  B2  C2  D6  E6

## one to many

In [26]:
df1 = make_df('ABC', range(3))
df2 = make_df('ADE', range(3))
df1.loc[1, 'A'] = 'A0'
display('df1', 'df2', 'pd.merge(df1, df2)')

df1
    A   B   C
0  A0  B0  C0
1  A0  B1  C1
2  A2  B2  C2

df2
    A   D   E
0  A0  D0  E0
1  A1  D1  E1
2  A2  D2  E2

pd.merge(df1, df2)
    A   B   C   D   E
0  A0  B0  C0  D0  E0
1  A0  B1  C1  D0  E0
2  A2  B2  C2  D2  E2

# many to many

In [27]:
df1 = make_df('ABC', range(3))
df2 = make_df('ADE', range(3))
df1.loc[1, 'A'] = 'A0'
df2.loc[1, 'A'] = df1.loc[1, 'A']
display('df1', 'df2', 'pd.merge(df1, df2)')

df1
    A   B   C
0  A0  B0  C0
1  A0  B1  C1
2  A2  B2  C2

df2
    A   D   E
0  A0  D0  E0
1  A0  D1  E1
2  A2  D2  E2

pd.merge(df1, df2)
    A   B   C   D   E
0  A0  B0  C0  D0  E0
1  A0  B0  C0  D1  E1
2  A0  B1  C1  D0  E0
3  A0  B1  C1  D1  E1
4  A2  B2  C2  D2  E2

### on

In [43]:
df1 = make_df('ABC', range(3))
df2 = make_df('ABE', range(3))
display('df1', 'df2', 'pd.merge(df1, df2, on=\'B\')')

df1
    A   B   C
0  A0  B0  C0
1  A1  B1  C1
2  A2  B2  C2

df2
    A   B   E
0  A0  B0  E0
1  A1  B1  E1
2  A2  B2  E2

pd.merge(df1, df2, on='B')
  A_x   B   C A_y   E
0  A0  B0  C0  A0  E0
1  A1  B1  C1  A1  E1
2  A2  B2  C2  A2  E2

### left_on and right_on

In [29]:
df1 = make_df('ABC', range(3))
df2 = make_df('FDE', range(3))
df1.loc[df2.index, 'B'] = df2['D']
display('df1', 'df2', 'pd.merge(df1, df2, left_on=\'B\', right_on=\'D\')')

df1
    A   B   C
0  A0  D0  C0
1  A1  D1  C1
2  A2  D2  C2

df2
    F   D   E
0  F0  D0  E0
1  F1  D1  E1
2  F2  D2  E2

pd.merge(df1, df2, left_on='B', right_on='D')
    A   B   C   F   D   E
0  A0  D0  C0  F0  D0  E0
1  A1  D1  C1  F1  D1  E1
2  A2  D2  C2  F2  D2  E2

## index joining. left_index and right_index

In [30]:
df1 = make_df('ABC', range(3))
df2 = make_df('DEF', range(3))
display('df1', 'df2', 'pd.merge(df1, df2, left_index=True, right_index=True)', 'df1.join(df2)')

df1
    A   B   C
0  A0  B0  C0
1  A1  B1  C1
2  A2  B2  C2

df2
    D   E   F
0  D0  E0  F0
1  D1  E1  F1
2  D2  E2  F2

pd.merge(df1, df2, left_index=True, right_index=True)
    A   B   C   D   E   F
0  A0  B0  C0  D0  E0  F0
1  A1  B1  C1  D1  E1  F1
2  A2  B2  C2  D2  E2  F2

df1.join(df2)
    A   B   C   D   E   F
0  A0  B0  C0  D0  E0  F0
1  A1  B1  C1  D1  E1  F1
2  A2  B2  C2  D2  E2  F2

# Specifying Set Arithmetic for Joins

In [31]:
df1 = pd.DataFrame({'candidate': ['Mary', 'Steve', 'Jane'],
                    'party': ['R', 'I', 'D']})
df2 = pd.DataFrame({'candidate': ['Joe', 'Mary', 'Steve'],
                    'votes': [1000, 800, 500]})
display('df1', 'df2', 'pd.merge(df1, df2, how=\'inner\')', 'pd.merge(df1, df2, how=\'outer\')', 'pd.merge(df1, df2, how=\'left\')', 'pd.merge(df1, df2, how=\'right\')')

df1
  candidate party
0      Mary     R
1     Steve     I
2      Jane     D

df2
  candidate  votes
0       Joe   1000
1      Mary    800
2     Steve    500

pd.merge(df1, df2, how='inner')
  candidate party  votes
0      Mary     R    800
1     Steve     I    500

pd.merge(df1, df2, how='outer')
  candidate party   votes
0      Jane     D     NaN
1       Joe   NaN  1000.0
2      Mary     R   800.0
3     Steve     I   500.0

pd.merge(df1, df2, how='left')
  candidate party  votes
0      Mary     R  800.0
1     Steve     I  500.0
2      Jane     D    NaN

pd.merge(df1, df2, how='right')
  candidate party  votes
0       Joe   NaN   1000
1      Mary     R    800
2     Steve     I    500

## overlapping column names

In [32]:
df1 = pd.DataFrame({'candidate': ['Mary', 'Steve', 'Jane'],
                    'votes': ['50%', '30%', '20%']})
df2 = pd.DataFrame({'candidate': ['Joe', 'Mary', 'Steve'],
                    'votes': [1000, 800, 500]})
display('df1', 'df2', 'pd.merge(df1, df2, on=\'candidate\')', 'pd.merge(df1, df2, on=\'candidate\', suffixes=(\'_percent\', \'_total\'))')

df1
  candidate votes
0      Mary   50%
1     Steve   30%
2      Jane   20%

df2
  candidate  votes
0       Joe   1000
1      Mary    800
2     Steve    500

pd.merge(df1, df2, on='candidate')
  candidate votes_x  votes_y
0      Mary     50%      800
1     Steve     30%      500

pd.merge(df1, df2, on='candidate', suffixes=('_percent', '_total'))
  candidate votes_percent  votes_total
0      Mary           50%          800
1     Steve           30%          500